# 07 · Asset Modularity and Instancing

> Companion notebook to `docs/asset-modularity-instancing/` from the Learn OpenUSD curriculum.

## Learning objectives
- Explain how OpenUSD empowers asset modularity through composition arcs and layer stacks.
- Understand scenegraph instancing: instanceable metadata, implicit prototypes, and instance proxies.
- Apply nested scenegraph instancing for large-scale scenes.
- Refine scenegraph instances via deinstancing, variant sets, hierarchical refinement (primvars, xformOps, visibility), ad hoc arcs, and broadcasted refinement (inherit / specialize).
- Author point instancing using `UsdGeom.PointInstancer` (prototypes, protoIndices, positions, orientations, scales).
- Refine point instances with `inactiveIds`, `invisibleIds`, promotion, and vertex-interpolated primvars.
- Compare scenegraph instancing vs. point instancing trade-offs.

## How to use this notebook
Run cells top-to-bottom. Generated files land in `./_assets/` next to this notebook. Safe to delete that folder any time.

In [1]:
import sys
print("Python:", sys.version.split()[0])
from pxr import Usd, Sdf, Gf, UsdGeom, UsdShade
print("pxr import OK.")
from pathlib import Path
NB_DIR = Path.cwd()
(NB_DIR / "_assets").mkdir(exist_ok=True)
print("_assets/ ready at:", NB_DIR / "_assets")

Python: 3.12.9
pxr import OK.
_assets/ ready at: /Users/stranzersweb/Projects/LearnOpenUSD/notebooks/.exec/07_asset_modularity_instancing/_assets


In [2]:
try:
    from lousd.utils.helperfunctions import create_new_stage
    from lousd.utils.visualization import DisplayUSD, DisplayCode
    print("lousd helpers loaded.")
    _HAVE_LOUSD = True
except ImportError:
    import os
    from pathlib import Path as _P
    def create_new_stage(relative_file_path: str):
        from pxr import Usd, Sdf
        ident = os.path.join(os.getcwd(), relative_file_path)
        found = Sdf.Layer.Find(identifier=ident)
        if found:
            return Usd.Stage.Open(found.identifier)
        return Usd.Stage.CreateNew(relative_file_path)
    def DisplayUSD(path, show_usd_code=True):
        from pxr import Usd
        print(f"--- {path} ---")
        print(Usd.Stage.Open(str(path)).ExportToString(addSourceFileComment=False))
    def DisplayCode(path):
        print(f"--- {path} ---")
        print(_P(path).read_text())
    print("Using fallback helpers.")
    _HAVE_LOUSD = False

lousd helpers loaded.


In [3]:
# Safely copy any available instancing exercise assets into _assets/.
import shutil
from pathlib import Path
SRC_EX = Path.cwd().parent / "docs" / "exercise_content" / "instancing"
DST_EX = Path.cwd() / "_assets" / "instancing_exercises"
try:
    if SRC_EX.exists():
        if DST_EX.exists():
            print("Exercise assets already staged at", DST_EX)
        else:
            shutil.copytree(SRC_EX, DST_EX)
            print("Copied exercise assets to", DST_EX)
    else:
        print("Exercise source not found at", SRC_EX, "- notebook will run without it.")
except Exception as e:
    print("Could not stage exercise assets (safe to ignore):", e)

Exercise source not found at /Users/stranzersweb/Projects/LearnOpenUSD/notebooks/.exec/docs/exercise_content/instancing - notebook will run without it.


## 7.1 Asset modularity

Modularity is the practice of building complex scenes out of small, self-contained, reusable assets. OpenUSD enables this through **composition arcs** (references, payloads, inherits, specializes, variants) stacked onto **layer stacks** (a root layer + its sublayers). Strength ordering is LIVERPS: **L**ocal, **I**nherits, **V**ariant Sets, **R**elocates, **R**eferences, **P**ayloads, **S**pecializes.

The benefits are the same ones you get from "DRY" in software: validation/standardization (the rack validated in one scenario works in another), efficiency through centralized editing (change one asset, update every scene), and system performance (layers and prims can be shared in memory).

### 7.1.1 Building a tiny modular asset library

We will hand-build a pocket-sized version of the warehouse asset library used in the course: a `CubeBox` component, a `BoxPallet` assembly that references the box, and a `Rack` assembly that references the pallet. Each asset is its own USD layer, which is what makes them modular.

In [4]:
from pxr import Usd, UsdGeom, Sdf, Gf
from pathlib import Path

ASSETS = Path.cwd() / "_assets"

# --- Component: CubeBox ---
box_stage = create_new_stage("_assets/CubeBox.usda")
box_stage.SetDefaultPrim(UsdGeom.Xform.Define(box_stage, "/CubeBox").GetPrim())
Usd.ModelAPI(box_stage.GetPrimAtPath("/CubeBox")).SetKind("component")
cube = UsdGeom.Cube.Define(box_stage, "/CubeBox/Geom")
cube.CreateSizeAttr(26.0)
box_stage.GetRootLayer().Save()

# --- Assembly: BoxPallet (references 4 CubeBoxes) ---
pallet_stage = create_new_stage("_assets/BoxPallet.usda")
pallet_stage.SetDefaultPrim(UsdGeom.Xform.Define(pallet_stage, "/BoxPallet").GetPrim())
Usd.ModelAPI(pallet_stage.GetPrimAtPath("/BoxPallet")).SetKind("assembly")
for i, (x, y) in enumerate([(0, 0), (30, 0), (0, 30), (30, 30)]):
    box_prim = pallet_stage.DefinePrim(f"/BoxPallet/Box_{i:02d}", "Xform")
    box_prim.GetReferences().AddReference("./CubeBox.usda")
    UsdGeom.Xformable(box_prim).AddTranslateOp().Set(Gf.Vec3d(x, y, 0))
pallet_stage.GetRootLayer().Save()

# --- Assembly: Rack (references 2 BoxPallets) ---
rack_stage = create_new_stage("_assets/Rack.usda")
rack_stage.SetDefaultPrim(UsdGeom.Xform.Define(rack_stage, "/Rack").GetPrim())
Usd.ModelAPI(rack_stage.GetPrimAtPath("/Rack")).SetKind("assembly")
for i, z in enumerate([0, 100]):
    p = rack_stage.DefinePrim(f"/Rack/Pallet_{i}", "Xform")
    p.GetReferences().AddReference("./BoxPallet.usda")
    UsdGeom.Xformable(p).AddTranslateOp().Set(Gf.Vec3d(0, 0, z))
rack_stage.GetRootLayer().Save()

for p in ["_assets/CubeBox.usda", "_assets/BoxPallet.usda", "_assets/Rack.usda"]:
    print(p, "-", (Path.cwd() / p).stat().st_size, "bytes")

_assets/CubeBox.usda - 152 bytes
_assets/BoxPallet.usda - 889 bytes
_assets/Rack.usda - 491 bytes


In [5]:
DisplayUSD("_assets/Rack.usda")

**What just happened:** we authored three layers. `Rack.usda` references `BoxPallet.usda`, which in turn references `CubeBox.usda`. Each asset is its own file, versionable and reusable. This is asset modularity at work: a single edit to `CubeBox.usda` automatically propagates up to every scene that uses the rack.

## 7.2 Authoring scenegraph instancing

**Scenegraph instancing** (sometimes called "native" instancing) lets OpenUSD turn repeated prim sub-hierarchies into shared **prototypes**. Four terms:

- **Prototype** — a unique shared sub-structure (implicit, created by USD).
- **Instance** — a repetition of a prototype in the scene.
- **Instanceable prim (Instance prim)** — the mutable root of an instance, tagged with `instanceable = true`.
- **Instance proxy** — a read-only stand-in for a prototype's descendants, addressable by path.

The mantra is **"explicit instances, implicit prototypes"**: you tag candidate prims with `instanceable = true`; OpenUSD figures out prototypes from the composition arcs underneath them. Instancing is driven by composition arcs — tagging a prim with no arcs does nothing.

### 7.2.1 Enabling `instanceable` and observing prototypes

In [6]:
from pxr import Usd, UsdGeom, Gf, UsdUtils

scenario = create_new_stage("_assets/07_scenario.usda")
world = UsdGeom.Xform.Define(scenario, "/World")
scenario.SetDefaultPrim(world.GetPrim())
warehouse = UsdGeom.Xform.Define(scenario, "/World/Warehouse").GetPrim()

# Reference the Rack assembly 5 times and lay them out in a row.
for i in range(5):
    rack = scenario.DefinePrim(f"/World/Warehouse/Rack_{i:02d}", "Xform")
    rack.GetReferences().AddReference("./Rack.usda")
    UsdGeom.Xformable(rack).AddTranslateOp().Set(Gf.Vec3d(i * 200.0, 0, 0))

scenario.GetRootLayer().Save()

stats_before = UsdUtils.ComputeUsdStageStats(scenario)
print("Before instancing:")
print("  totalPrimCount    =", stats_before.get("totalPrimCount"))
print("  totalInstanceCount=", stats_before.get("totalInstanceCount"))
print("  prototypeCount    =", stats_before.get("prototypeCount"))

Before instancing:
  totalPrimCount    = 97
  totalInstanceCount= 0
  prototypeCount    = 0


In [7]:
# Now tag every Rack_* prim as instanceable. Because each has a reference arc
# to Rack.usda, USD can create a single shared prototype for all 5 of them.
for i in range(5):
    rack = scenario.GetPrimAtPath(f"/World/Warehouse/Rack_{i:02d}")
    rack.SetInstanceable(True)

scenario.GetRootLayer().Save()

stats_after = UsdUtils.ComputeUsdStageStats(scenario)
print("After instancing:")
print("  totalPrimCount    =", stats_after.get("totalPrimCount"))
print("  totalInstanceCount=", stats_after.get("totalInstanceCount"))
print("  prototypeCount    =", stats_after.get("prototypeCount"))
print("\nPrototype prims:")
for proto in scenario.GetPrototypes():
    print(" ", proto.GetPath(), "-> used by", len(proto.GetInstances()), "instances")

After instancing:
  totalPrimCount    = 26
  totalInstanceCount= 5
  prototypeCount    = 1

Prototype prims:
  /__Prototype_1 -> used by 5 instances


**What just happened:** after setting `instanceable = true` on each rack, the total prim count dropped dramatically — USD built a single implicit `/__Prototype_1` from the reference arc on Rack.usda, and the five racks became instances of it. `GetPrototypes()` / `GetInstances()` give you programmatic access to the prototype graph.

### 7.2.2 The trick question: instancing without composition arcs

Setting `instanceable = true` only matters if the prim has composition arcs for OpenUSD to derive a prototype from. Local opinions alone don't count.

In [8]:
from pxr import Usd, UsdGeom, Gf, UsdUtils
trick = create_new_stage("_assets/07_trick.usda")
w = UsdGeom.Xform.Define(trick, "/Warehouse")
trick.SetDefaultPrim(w.GetPrim())
for i in range(3):
    # No references/payloads/inherits/variants -- just local def.
    crate = UsdGeom.Xform.Define(trick, f"/Warehouse/Crate_{i:02d}")
    crate.AddTranslateOp().Set(Gf.Vec3d(i * 10.0, 20.0, 0))
    crate.GetPrim().SetInstanceable(True)
trick.GetRootLayer().Save()

stats = UsdUtils.ComputeUsdStageStats(trick)
print("prototypeCount =", stats.get("prototypeCount"), "(expected: 0)")
print("totalInstanceCount =", stats.get("totalInstanceCount"), "(expected: 0)")

prototypeCount = 0 (expected: 0)
totalInstanceCount = 0 (expected: 0)


**What just happened:** zero prototypes despite three `instanceable = true` prims. Lesson: instancing rides on top of composition arcs.

### 7.2.3 Nested instancing

Nested instancing means an instance's subgraph itself contains more instances. Common patterns: instanced assemblies containing instanced components, or instanced components referencing instanced material networks. Nesting lets you push instancing higher up the hierarchy for larger performance wins, at the cost of authoring flexibility — the higher you go, the more local overrides inside the instance are discarded.

If we tag both the `Rack_*` prims **and** the pallets inside the rack as instanceable, USD gets nested prototypes.

In [9]:
# Re-author Rack.usda so the inner BoxPallet references are also instanceable.
from pxr import Usd
rack_stage2 = Usd.Stage.Open("_assets/Rack.usda")
for child in rack_stage2.GetPrimAtPath("/Rack").GetChildren():
    child.SetInstanceable(True)
rack_stage2.GetRootLayer().Save()

# Reload the scenario so the change propagates through composition.
scenario = Usd.Stage.Open("_assets/07_scenario.usda")
nested_stats = UsdUtils.ComputeUsdStageStats(scenario)
print("Nested instancing stats:")
print("  totalPrimCount    =", nested_stats.get("totalPrimCount"))
print("  totalInstanceCount=", nested_stats.get("totalInstanceCount"))
print("  prototypeCount    =", nested_stats.get("prototypeCount"))
print("\nPrototypes:")
for proto in scenario.GetPrototypes():
    print(" ", proto.GetPath())

Nested instancing stats:
  totalPrimCount    = 19
  totalInstanceCount= 7
  prototypeCount    = 2

Prototypes:
  /__Prototype_1
  /__Prototype_2


**What just happened:** USD now has a Rack prototype whose subgraph itself instances a BoxPallet prototype. In a real warehouse (25 racks × 3 pallets × 18 boxes), nested instancing in the course exercise drops a 44,408-prim stage to ~1,482 prims.

## 7.3 Refining scenegraph instances

The key rules of **instance editability**:

1. **Prototypes are a runtime data model** — not editable.
2. **Instance proxies are not editable** — local opinions on them are discarded (or error out).
3. **Instanceable prims are editable** — so you can at least position, scale, and activate each instance.

"Refinement" is the set of techniques that still let you introduce variety into an instanced scene without losing all the benefits of instancing: **deinstancing**, **variant sets**, **hierarchical refinement** (primvars / xformOps / visibility on ancestors), **ad hoc arcs**, and **broadcasted refinement** via inherit/specialize arcs.

### 7.3.1 Instance editability (the hard rule)

In [10]:
from pxr import Usd, UsdGeom, Gf, Tf
scenario = Usd.Stage.Open("_assets/07_scenario.usda")

rack = scenario.GetPrimAtPath("/World/Warehouse/Rack_00")
print("Rack_00 is instanceable:", rack.IsInstanceable(), "is instance:", rack.IsInstance())

# Valid: translate the instanceable prim itself.
xformable = UsdGeom.Xformable(rack)
existing_ops = xformable.GetOrderedXformOps()
translate_op = next((op for op in existing_ops if op.GetOpType() == UsdGeom.XformOp.TypeTranslate), None)
if translate_op is None:
    translate_op = xformable.AddTranslateOp()
translate_op.Set(Gf.Vec3d(0, 0, 50.0))
print("Moved Rack_00 up 50 units (allowed on instanceable prim).")

# Invalid: try to author on an instance proxy inside the instance.
proxy_path = "/World/Warehouse/Rack_00/Pallet_0"
proxy = scenario.GetPrimAtPath(proxy_path)
print("\nPallet_0 is instance proxy:", proxy.IsInstanceProxy())
try:
    UsdGeom.Xformable(proxy).AddTranslateOp().Set(Gf.Vec3d(999, 0, 0))
    print("Unexpected: edit succeeded on instance proxy.")
except Tf.ErrorException as e:
    print("Expected error on instance-proxy edit:")
    print(" ", str(e).splitlines()[0])

Rack_00 is instanceable: True is instance: True
Moved Rack_00 up 50 units (allowed on instanceable prim).

Pallet_0 is instance proxy: True
Expected error on instance-proxy edit:
  


**What just happened:** moving the instanceable root (`Rack_00`) is fine — that's how you place instances uniquely. Authoring a property on `/World/Warehouse/Rack_00/Pallet_0` raises `Tf.ErrorException` because it's an instance proxy backed by a shared prototype.

### 7.3.2 Deinstancing refinement

The simplest refinement: turn `instanceable` off for one copy to regain full editability. You lose some optimization on that copy but still benefit from composition-level reuse. Good for one-off overrides; wasteful if you'd have to repeat it on many instances.

In [11]:
rack = scenario.GetPrimAtPath("/World/Warehouse/Rack_00")
rack.SetInstanceable(False)
print("Rack_00 is now instanceable:", rack.IsInstanceable())

# Now we can freely author on what used to be an instance proxy.
pallet = scenario.GetPrimAtPath("/World/Warehouse/Rack_00/Pallet_0")
UsdGeom.Imageable(pallet).MakeInvisible()
print("Hid /World/Warehouse/Rack_00/Pallet_0 via a local visibility opinion.")

stats = UsdUtils.ComputeUsdStageStats(scenario)
print("\nStats after deinstancing Rack_00:")
print("  totalPrimCount     =", stats.get("totalPrimCount"))
print("  totalInstanceCount =", stats.get("totalInstanceCount"))
print("  prototypeCount     =", stats.get("prototypeCount"))

# Restore for the next sections.
rack.SetInstanceable(True)

Rack_00 is now instanceable: False
Hid /World/Warehouse/Rack_00/Pallet_0 via a local visibility opinion.

Stats after deinstancing Rack_00:
  totalPrimCount     = 21
  totalInstanceCount = 8
  prototypeCount     = 2


True

**What just happened:** setting `instanceable = False` lets us author a visibility override on a formerly-proxy prim. The prim count goes up (more uniqueness) but the edit works.

### 7.3.3 Hierarchical refinement using primvars

Primvars inherit down the prim hierarchy. Setting a primvar on the instanceable prim (or any ancestor) is a local edit on the instance — totally legal — and materials inside the instance's prototype can read it through `UsdPrimvarReader_*` shaders. No new prototypes get created, so it's the cheapest possible refinement for visual variety (color, "dirtiness", etc.).

In [12]:
from pxr import Sdf, UsdGeom
rack0 = scenario.GetPrimAtPath("/World/Warehouse/Rack_00")
rack2 = scenario.GetPrimAtPath("/World/Warehouse/Rack_02")

# Author primvars:cleanness on two instanceable racks -- the instances remain
# fully shared, and the one prototype can read this inherited primvar.
pv_api0 = UsdGeom.PrimvarsAPI(rack0)
pv0 = pv_api0.CreatePrimvar("cleanness", Sdf.ValueTypeNames.FloatArray, UsdGeom.Tokens.constant)
pv0.Set([0.8])

pv_api2 = UsdGeom.PrimvarsAPI(rack2)
pv2 = pv_api2.CreatePrimvar("cleanness", Sdf.ValueTypeNames.FloatArray, UsdGeom.Tokens.constant)
pv2.Set([0.4])

stats = UsdUtils.ComputeUsdStageStats(scenario)
print("Prim count unchanged:", stats.get("totalPrimCount"))
print("Prototype count unchanged:", stats.get("prototypeCount"))
print("Instance count unchanged:", stats.get("totalInstanceCount"))
print("Cleanness on Rack_00:", pv0.Get())
print("Cleanness on Rack_02:", pv2.Get())

Prim count unchanged: 19
Prototype count unchanged: 2
Instance count unchanged: 7
Cleanness on Rack_00: [0.8]
Cleanness on Rack_02: [0.4]


**What just happened:** both racks still point at the same prototype, yet each carries its own `primvars:cleanness`. Primvar inheritance lets shaders inside the prototype pick it up — free variety.

### 7.3.4 Refinement using variant sets

Variant sets are part of composition, so they participate in prototype derivation. If the asset defines a `statusVariant` with `default` / `allocated` / `shipped` selections, switching the selection on an instanceable prim produces a **new prototype** shared by all instances with that selection. More efficient than deinstancing when the same override applies to many instances.

In [13]:
# Give the CubeBox asset a tiny variant set so our instances can switch between them.
from pxr import Usd, UsdGeom, Sdf
box_stage = Usd.Stage.Open("_assets/CubeBox.usda")
box_prim = box_stage.GetPrimAtPath("/CubeBox")
vset = box_prim.GetVariantSets().AddVariantSet("statusVariant")
for name in ["default", "allocated", "shipped"]:
    vset.AddVariant(name)
vset.SetVariantSelection("default")
with vset.GetVariantEditContext():
    box_prim.CreateAttribute("status", Sdf.ValueTypeNames.String).Set("DEFAULT")
vset.SetVariantSelection("allocated")
with vset.GetVariantEditContext():
    box_prim.CreateAttribute("status", Sdf.ValueTypeNames.String).Set("ALLOCATED")
vset.SetVariantSelection("shipped")
with vset.GetVariantEditContext():
    box_prim.CreateAttribute("status", Sdf.ValueTypeNames.String).Set("SHIPPED")
vset.SetVariantSelection("default")
box_stage.GetRootLayer().Save()

# Build a small variant-demo scene that references CubeBox twice with different selections.
vs = create_new_stage("_assets/07_variant_demo.usda")
w = UsdGeom.Xform.Define(vs, "/World")
vs.SetDefaultPrim(w.GetPrim())
for i, selection in enumerate(["default", "allocated", "shipped", "allocated"]):
    p = vs.DefinePrim(f"/World/Box_{i:02d}", "Xform")
    p.GetReferences().AddReference("./CubeBox.usda")
    p.GetVariantSets().GetVariantSet("statusVariant").SetVariantSelection(selection)
    p.SetInstanceable(True)
vs.GetRootLayer().Save()

stats = UsdUtils.ComputeUsdStageStats(vs)
print("Variant-refined stage prototypes:", stats.get("prototypeCount"), "(expected: 3 -- one per unique selection)")
print("Instances:", stats.get("totalInstanceCount"))
for i in range(4):
    p = vs.GetPrimAtPath(f"/World/Box_{i:02d}")
    print(" ", p.GetPath(), "status=", p.GetAttribute("status").Get())

Variant-refined stage prototypes: 3 (expected: 3 -- one per unique selection)
Instances: 4
  /World/Box_00 status= DEFAULT
  /World/Box_01 status= ALLOCATED
  /World/Box_02 status= SHIPPED
  /World/Box_03 status= ALLOCATED


**What just happened:** four instanceable boxes picking three different variant selections produce three prototypes (one per unique selection). The two `allocated` boxes share a prototype. Much cheaper than deinstancing all four.

### 7.3.5 Ad hoc arcs refinement

Add a new composition arc (e.g. an internal reference to a "DamagedStamp" layer) directly to an instanceable prim. USD creates a new prototype for that arc configuration and any other instance with the same arc setup will share it. Useful when several instances need the same ad hoc override.

In [14]:
# Tiny "stamp" layer we can reference ad-hoc onto an instanceable prim.
stamp = create_new_stage("_assets/DamagedStamp.usda")
over = stamp.OverridePrim("/DamagedStamp")
over.CreateAttribute("stampText", Sdf.ValueTypeNames.String).Set("DAMAGED")
stamp.SetDefaultPrim(over)
stamp.GetRootLayer().Save()

# Reopen the scenario and stamp two racks.
scenario = Usd.Stage.Open("_assets/07_scenario.usda")
for path in ["/World/Warehouse/Rack_01", "/World/Warehouse/Rack_03"]:
    p = scenario.GetPrimAtPath(path)
    p.GetReferences().AddReference("./DamagedStamp.usda")
scenario.GetRootLayer().Save()

stats = UsdUtils.ComputeUsdStageStats(scenario)
print("Prototypes after ad hoc arcs:", stats.get("prototypeCount"))
print("Instances:", stats.get("totalInstanceCount"))
for proto in scenario.GetPrototypes():
    print(" ", proto.GetPath(), "used by", len(proto.GetInstances()))

Prototypes after ad hoc arcs: 3
Instances: 9
  /__Prototype_1 used by 4
  /__Prototype_2 used by 3
  /__Prototype_3 used by 2


**What just happened:** Rack_01 and Rack_03 now have an extra reference to the stamp layer. Their composition arc signature is different from Rack_00/02/04, so USD creates a second prototype for them that they share between each other.

### 7.3.6 Broadcasted refinement (inherit / specialize)

Inherit and specialize arcs have a special broadcasting property: an opinion authored on the inherited namespace gets broadcast to every prim that inherits/specializes from it. Combined with instancing, you can author one override on a class prim and have USD make a new shared prototype for every instance that inherits from it. We won't wire the full shader here; this cell just shows the API shape.

In [15]:
bc = create_new_stage("_assets/07_broadcast.usda")
w = UsdGeom.Xform.Define(bc, "/World")
bc.SetDefaultPrim(w.GetPrim())

# A class prim acts as a specialize target.
class_prim = bc.CreateClassPrim("/World/_PalletBox")

# Three boxes that all specialize from /World/_PalletBox.
for i in range(3):
    p = bc.DefinePrim(f"/World/Box_{i}", "Xform")
    p.GetReferences().AddReference("./CubeBox.usda")
    p.GetSpecializes().AddSpecialize("/World/_PalletBox")
    p.SetInstanceable(True)

# Broadcast a tag onto the class prim -- every specialize-target instance picks it up.
class_prim.CreateAttribute("damageTag", Sdf.ValueTypeNames.String).Set("DAMAGED")
bc.GetRootLayer().Save()

for i in range(3):
    p = bc.GetPrimAtPath(f"/World/Box_{i}")
    print(p.GetPath(), "damageTag:", p.GetAttribute("damageTag").Get(),
          "is instance:", p.IsInstance())
print("Prototypes:", UsdUtils.ComputeUsdStageStats(bc).get("prototypeCount"))

/World/Box_0 damageTag: DAMAGED is instance: True
/World/Box_1 damageTag: DAMAGED is instance: True
/World/Box_2 damageTag: DAMAGED is instance: True
Prototypes: 1


**What just happened:** three instanceable boxes all specialize from `/World/_PalletBox`. One opinion on the class prim broadcasts to all three and they can still share a prototype derived from their (identical) composition signatures.

## 7.4 Authoring point instancing

**Point instancing** is schema-based, not composition-based. A `UsdGeom.PointInstancer` holds:

- `prototypes` — a **relationship** to explicit prototype prim hierarchies (typically under an `over Scope "Prototypes"`).
- `protoIndices` — an int array mapping each point to a prototype index.
- `positions` — per-point positions in the instancer's local space.
- Optional `orientations`, `scales`, `ids`, `velocities`, `angularVelocities`.

Point instancing trades readability for raw scale — you can pack hundreds of thousands of instances into a few vectorized arrays. It's the right tool for leaves on trees, packing peanuts, debris, crowds, etc., where individual prim addressability isn't worth its overhead.

In [16]:
from pxr import Usd, UsdGeom, Sdf, Gf, Vt
import random

pi_stage = create_new_stage("_assets/07_pointinstancer.usda")
world = UsdGeom.Xform.Define(pi_stage, "/World")
pi_stage.SetDefaultPrim(world.GetPrim())

pi = UsdGeom.PointInstancer.Define(pi_stage, "/World/Scatter")

# Explicit prototypes: two CubeBox references under an over Scope.
protos_path = Sdf.Path("/World/Scatter/Prototypes")
pi_stage.OverridePrim(protos_path)  # over Scope
for i, tag in enumerate(["A", "B"]):
    proto = pi_stage.DefinePrim(protos_path.AppendChild(f"Box_{tag}"), "Xform")
    proto.GetReferences().AddReference("./CubeBox.usda")

# Wire the prototypes relationship.
pi.CreatePrototypesRel().SetTargets([
    protos_path.AppendChild("Box_A"),
    protos_path.AppendChild("Box_B"),
])

# Scatter 200 instances on a 20x10 grid; pick prototype 0 or 1 randomly.
random.seed(0)
N = 200
positions = Vt.Vec3fArray([Gf.Vec3f((i % 20) * 40.0, (i // 20) * 40.0, 0.0) for i in range(N)])
proto_indices = Vt.IntArray([random.randint(0, 1) for _ in range(N)])
orientations = Vt.QuathArray([Gf.Quath(1, 0, 0, 0)] * N)
scales = Vt.Vec3fArray([Gf.Vec3f(1, 1, 1)] * N)

pi.CreatePositionsAttr(positions)
pi.CreateProtoIndicesAttr(proto_indices)
pi.CreateOrientationsAttr(orientations)
pi.CreateScalesAttr(scales)
pi_stage.GetRootLayer().Save()

print("PointInstancer created with", len(positions), "points and",
      len(pi.GetPrototypesRel().GetTargets()), "prototypes.")

PointInstancer created with 200 points and 2 prototypes.


In [17]:
DisplayUSD("_assets/07_pointinstancer.usda")  # verbose because of the arrays

**What just happened:** a single `PointInstancer` prim holds 200 instances, two prototypes, and the arrays needed to place each one. Transforms per instance compose as: `prototype root xform` → `scales[i]` → `orientations[i]` → `positions[i]` → the PointInstancer's own prim xform.

## 7.5 Refining point instances

Refinement options for PointInstancers:

- **Vertex-interpolated primvars** — one value per instance; good for per-instance color / cleanness.
- **Add new prototypes** — define another prim, append it to `prototypes`, update `protoIndices`.
- **Visibility / deactivation** — `invisibleIds` (animatable, per-frame) or `inactiveIds` metadata (sparse list-editable prune).
- **Promotion** — deactivate a point and replace it with a full referenced prim you can freely override, using `ComputeInstanceTransformsAtTime` to copy the transform.

In [18]:
from pxr import Usd, UsdGeom, Sdf, Gf, Vt
pi_stage = Usd.Stage.Open("_assets/07_pointinstancer.usda")
pi = UsdGeom.PointInstancer.Get(pi_stage, "/World/Scatter")

# 1) Deactivate a specific instance id.
pi.DeactivateId(42)
print("Deactivated instance id 42.")

# 2) Author a vertex-interpolated cleanness primvar (one value per point).
import random
random.seed(1)
pv_api = UsdGeom.PrimvarsAPI(pi)
cleanness = pv_api.CreatePrimvar(
    "cleanness", Sdf.ValueTypeNames.FloatArray, UsdGeom.Tokens.vertex
)
cleanness.Set([random.uniform(0.3, 1.0) for _ in range(200)])
print("primvars:cleanness set on PointInstancer (vertex interpolation).")

# 3) Promote instance id 7 into a standalone prim.
xforms = pi.ComputeInstanceTransformsAtTime(
    Usd.TimeCode.Default(),
    Usd.TimeCode.Default(),
    applyMask=UsdGeom.PointInstancer.IgnoreMask,
)
m = xforms[7]
promoted = pi_stage.DefinePrim("/World/PromotedBox_07", "Xform")
promoted.GetReferences().AddReference("./CubeBox.usda")
translation = m.ExtractTranslation()
rotation = m.ExtractRotation()
scale = Gf.Vec3d(*(v.GetLength() for v in m.ExtractRotationMatrix()))
xf = UsdGeom.Xformable(promoted)
xf.AddTranslateOp().Set(translation)
xf.AddOrientOp().Set(Gf.Quatf(rotation.GetQuat()))
xf.AddScaleOp().Set(scale)
pi.DeactivateId(7)
print("Promoted id 7 to", promoted.GetPath(), "at", translation)

pi_stage.GetRootLayer().Save()

Deactivated instance id 42.
primvars:cleanness set on PointInstancer (vertex interpolation).
Promoted id 7 to /World/PromotedBox_07 at (280, 0, 0)


True

**What just happened:** we deactivated id 42 (pruned), set a per-instance primvar to introduce shading variety, and promoted id 7 into a full-fledged overridable prim after deactivating it. `ComputeInstanceTransformsAtTime` returns the full per-instance matrix including the prototype root and velocities — the robust way to extract a promoted position.

## 7.6 Instancing FAQ: scenegraph vs. point

| | **Scenegraph instancing** | **Point instancing** |
|---|---|---|
| Mechanism | Composition-based | Schema-based (`UsdGeomPointInstancer`) |
| Prototypes | Implicit, derived from composition arcs | Explicit prim hierarchies referenced by a relationship |
| Addressability | Each instance has a path; proxies queryable by path | Each instance identified by integer id/index |
| Editability | Instanceable prim mutable; proxies read-only | Must re-author full arrays; sparse overrides limited |
| Deinstance | Transparent (`SetInstanceable(False)`) | Invasive (promote + deactivate) |
| Sweet spot | Reusing complex components (rigs, assemblies, racks) | Massive numbers of simple repetitions (leaves, peanuts, debris) |

The two can be nested: scenegraph instances can contain PointInstancers and vice versa. Pick the mechanism that matches the *grain* of your repetition.

## Key takeaways

- **Modularity comes from composition arcs + layer stacks.** DRY applies to 3D scenes just like code.
- **Scenegraph instancing is declarative deduplication of composition.** You mark prims `instanceable`; USD builds implicit prototypes from arcs.
- **"Explicit instances, implicit prototypes."** No composition arcs, no prototypes — instancing on locally-defined prims is a no-op.
- **Three rules of editability:** prototypes not editable, instance proxies not editable, instanceable prims editable.
- **Refinement techniques** let you reintroduce variety: deinstance, hierarchical refinement (primvars/xformOps/visibility), variant sets, ad hoc arcs, broadcasted refinement via inherit/specialize. Each trades some optimization for some authoring flexibility.
- **Point instancing** packs massive repetition into vectorized arrays. Refine with primvars, `inactiveIds`/`invisibleIds`, or promotion.
- **Nested instancing** (scenegraph-in-scenegraph, point-in-scenegraph, scenegraph-in-point) is a real and powerful production technique — use it where complexity allows.

## Next up

Continue with **`08_data_exchange.ipynb`**, where we look at how OpenUSD interchanges data with other DCCs and pipelines.